# Technical PDF RAG Assistant with Hybrid Retrieval and

Load functions from src folder.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

## Load Files:

Testing parse_pdf_with_metadata function using langchain_community PyMuPDFLoader.

In [ ]:
from src.loaders import parse_pdf_folder

folder_path = 'data/raw/'
files = parse_pdf_folder(folder_path)
print("Total pages: ", len(files))

In [ ]:
print("First page content:\n", files[0].page_content)

In [ ]:
print("First page metadata:\n", files[0].metadata)

## Data Cleaning:

In [ ]:
from src.preprocessing import clean_text
for i,text in enumerate(files):
    text.page_content = clean_text(text.page_content)

In [ ]:
print("First page cleaned content:\n", files[0].page_content)

## Chunking Data:

In [ ]:
from src.chunking import split_text

chunks = split_text(files)
print(chunks)

## Embedding:

In [ ]:
from src.embeddings import build_embedding_model

minilm_embedding_model = build_embedding_model("miniLM")
coher_embedding_model = build_embedding_model("cohere_v3")

## Indexing:

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
from src.config import ROOT_DIR

# If your notebook is inside the project root, this is enough:
load_dotenv(ROOT_DIR / ".env")

print(os.getenv("COHERE_API_KEY"))

In [ ]:
from src.vectorstore import build_database, save_faiss_index
vectorstore = build_database(chunks, minilm_embedding_model)
save_faiss_index(vectorstore, "faiss_minilm")

In [ ]:
from src.vectorstore import build_database, save_faiss_index
vectorstore = build_database(chunks, coher_embedding_model)
save_faiss_index(vectorstore, "cohere_v3")

## Load Index and Query

In [ ]:
from src.vectorstore import build_database, load_faiss_index
embedding_model = build_embedding_model("miniLM")
vectorstore = load_faiss_index("faiss_minilm", embedding_model)